# Task 4 — Deploy the Sentiment Analysis Demo

This walkthrough prepares the Task 1 sentiment model for a public demo as a **single Streamlit Community Cloud app**. The app loads the model artifact from this repository and predicts locally inside the app; it does not call a separate API.

**Deployment status:** the app and local tests are implemented. Public deployment is complete only after the app is created in Streamlit Community Cloud, opened at its public URL, and tested there. Do not record a public URL before those checks pass.

**Data flow:** a visitor submits a review in Streamlit → the app transforms it with the saved Task 1 vectorizer → the classifier predicts the sentiment → Streamlit displays the class and relative class scores. Submitted reviews are not intentionally stored. The model artifact and app code are public in this repository; do not commit private review data or credentials.

## 1. Files and prerequisites

- `models/task1_recommended_model.joblib` — fitted TF-IDF vectorizer, LinearSVC classifier, and class labels from Task 1.
- `deploy/streamlit_app.py` — public interface and local inference code.
- `deploy/requirements.txt` — pinned model and app dependencies installed by Streamlit Community Cloud.

The Task 1 model artifact uses scikit-learn serialization. The app pins the same scikit-learn, NumPy, SciPy, and joblib versions used to create the artifact to avoid incompatible model loading.

In [ ]:
from pathlib import Path
import joblib
import sys

working_path = Path.cwd().resolve()
ROOT = next(
    (path for path in (working_path, *working_path.parents)
     if (path / "deploy" / "streamlit_app.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / "models" / "task1_recommended_model.joblib"
assert MODEL_PATH.is_file(), f"Task 1 model artifact is missing: {MODEL_PATH}"
print(f"Project root: {ROOT}")
print(f"Model artifact: {MODEL_PATH} ({MODEL_PATH.stat().st_size:,} bytes)")
print(f"Streamlit app: {ROOT / 'deploy' / 'streamlit_app.py'}")

## 2. Test the Streamlit app locally

The local app loads the saved Task 1 model, submits an example review, checks that all three sentiment scores are displayed, and verifies blank input is rejected. It makes no hosted API calls and does not publish anything.

## 2.1 Read the actual app source code

The cell below displays the Streamlit app source directly from `deploy/`, so this walkthrough stays in sync with the implementation rather than maintaining a copied version of the app.

In [ ]:
from IPython.display import Markdown, display

deployment_sources = {
    "Streamlit app and local inference": ROOT / "deploy" / "streamlit_app.py",
    "Streamlit app dependencies": ROOT / "deploy" / "requirements.txt",
}
for title, source_path in deployment_sources.items():
    if not source_path.is_file():
        raise FileNotFoundError(f"Deployment source is missing: {source_path}")
    suffix = source_path.suffix.lstrip(".") or "text"
    source = source_path.read_text(encoding="utf-8")
    display(Markdown(f"### {title}\n\n`{source_path.relative_to(ROOT)}`\n\n```{suffix}\n{source}\n```"))

In [ ]:
from streamlit.testing.v1 import AppTest

streamlit_test = AppTest.from_file(
    str(ROOT / "deploy" / "streamlit_app.py"),
    default_timeout=30,
).run()
streamlit_test.text_area[0].set_value(
    "This product is excellent and works perfectly."
)
streamlit_test.button[0].click().run()
assert not streamlit_test.exception, [
    error.value for error in streamlit_test.exception
]
assert any(
    "Predicted sentiment:" in item.value for item in streamlit_test.subheader
)
print("The Streamlit app loaded the model and returned a prediction.")
print("Score note:", streamlit_test.caption[0].value)

### Interpreting the scores

The classifier is LinearSVC, which does not produce calibrated probabilities. The app applies softmax to its decision values only to make a relative three-class score display. The UI explicitly says these are **not calibrated probabilities**; do not present them as true confidence estimates.

## 3. Publish the Streamlit app

1. Confirm the public GitHub repository contains `deploy/streamlit_app.py`, `deploy/requirements.txt`, and `models/task1_recommended_model.joblib`. Do not include private customer data, API tokens, or `.streamlit/secrets.toml`.
2. Sign in to Streamlit Community Cloud with access to the repository and create a new app.
3. Select repository `alkilicayse87-pixel/project-brief-NLP-automated-customers-reviews`, branch `main`, and app file path `deploy/streamlit_app.py`.
4. Wait for the app to build. The model file is public and is loaded directly by the app; no backend URL or Streamlit secrets are required.

In [ ]:
# Confirm the compact fitted artifact is present for Cloud deployment.
assert MODEL_PATH.is_file(), f"Model artifact is missing: {MODEL_PATH}"
assert MODEL_PATH.stat().st_size < 1_000_000
print(f"Model artifact ready ({MODEL_PATH.stat().st_size:,} bytes).")

## 4. Verify the deployed app

Open the public URL Streamlit Community Cloud assigns to the app. Submit positive, negative, and neutral example reviews; confirm each prediction renders with three relative scores and the score caveat. Also verify blank input is rejected and a review near the 5,000-character limit can be submitted. Record the real public URL only after these checks pass.

In [ ]:
# Local model smoke test for examples from each sentiment class.
artifact = joblib.load(MODEL_PATH)
for review in (
    "I love this product; it works perfectly.",
    "This is an ordinary product with average features.",
    "I hate this product; it broke immediately.",
):
    features = artifact["vectorizer"].transform([review])
    predicted = artifact["classifier"].predict(features)[0]
    print(f"{predicted}: {review}")

## 5. Test and publish the Streamlit app

1. Run the local Streamlit test cell below and confirm a prediction is rendered.
2. Push the app, pinned dependencies, and public Task 1 model artifact to the GitHub repository connected to Streamlit Community Cloud.
3. In Streamlit Community Cloud, create an app from branch `main` and set its app file path to `deploy/streamlit_app.py`.
4. Wait for the build, open the public URL, and test positive, negative, neutral, blank, and long review inputs.
5. Record the actual Streamlit app URL in the README and project submission only after those checks pass.

Reviews are processed in the Streamlit app only after submission and are not intentionally stored. Do not enter personal or sensitive information. The public educational app has no authentication or rate limiting and is not a production service.

In [ ]:
# Run the app locally and submit a sample review.
from streamlit.testing.v1 import AppTest

streamlit_test = AppTest.from_file(
    str(ROOT / "deploy" / "streamlit_app.py"),
    default_timeout=30,
).run()
streamlit_test.text_area[0].set_value(
    "This product is excellent and works perfectly."
)
streamlit_test.button[0].click().run()
assert not streamlit_test.exception, [
    error.value for error in streamlit_test.exception
]
assert any(
    "Predicted sentiment:" in item.value
    for item in streamlit_test.subheader
)
print("Streamlit local prediction check passed.")

## Results and submission checklist

- [x] Task 1 fitted model artifact exists and loads locally.
- [x] Streamlit app loads the fitted Task 1 artifact and predicts locally.
- [x] App validates blank reviews and displays three relative class scores.
- [ ] Streamlit Community Cloud app is published and tested at its public URL.
- [ ] Public Streamlit URL is added to `README.md` and submitted with the project.

Until the public Streamlit app has been tested, Task 4's code is prepared but the deployment is not complete. There is no verified public URL to list yet.